# KAGGLE CHURN CHALLENGE

## Problem statement
Customer churn is a critical issue for subscription-based digital platforms, where losing a user directly impacts long-term revenue. On the platform under study, each user generates a sequence of interaction logs over time, including page visits, song plays, session information, and account actions. Each user is uniquely identified by a userId and can have multiple log entries that collectively represent their behavioral trajectory on the platform.

A user is labeled as churned if their activity history contains the page event "Cancellation Confirmation", which indicates that the user has completed the process of closing their account. Users who never visit this page are labeled as non-churners.
The training dataset contains full activity trajectories for users who have already churned as well as those who have remained active. The test dataset contains activity sequences from new users over the same time period, but none of them have yet reached the “Cancellation Confirmation” step—meaning their churn outcome is unknown.

The objective of this project is to predict whether a given user is likely to churn, based on their historical activity logs. Instead of relying solely on aggregated user-level features, the proposed approach aims to model each user’s behavioral trajectory over time. In this formulation, every user is represented as a temporal sequence of actions, and a model such as k-Nearest Neighbors (k-NN) or another sequence-similarity-based method will be used to compare a new user’s trajectory to those of known churners and non-churners. The underlying assumption is that users with similar behavioral patterns exhibit similar churn tendencies.

The challenge is therefore to design an appropriate representation of user trajectories, define a suitable distance or similarity measure between sequences, and train a model able to classify previously unseen users based on their activity evolution. The final goal is to provide an accurate, interpretable, and deployable churn prediction system that anticipates risk before the user explicitly cancels their subscription.

## Importation of useful libraries

In [103]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from skrub import GapEncoder

## Loading the dataset

In [104]:
dataset=pd.read_parquet("train.parquet")

In [105]:
data_copy=dataset.copy()


In [106]:
df=data_copy.copy()

## Looking at the variables in the dataset

In [107]:
df.head()

#for now I don't understand these variables: status,ts, auth, sessionId, IteminSession, page.values, time

,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
0,200,M,Shlok,paid,Johnson,1749042,1538352001000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",278,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,524.32934,Ich mache einen Spiegel - Dream Part 4,Popol Vuh,2018-10-01 00:00:01,2018-08-08 13:22:21
992,200,M,Shlok,paid,Johnson,1749042,1538352525000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",279,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,178.02404,Monster (Album Version),Skillet,2018-10-01 00:08:45,2018-08-08 13:22:21
1360,200,M,Shlok,paid,Johnson,1749042,1538352703000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",280,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,232.61995,Seven Nation Army,The White Stripes,2018-10-01 00:11:43,2018-08-08 13:22:21
1825,200,M,Shlok,paid,Johnson,1749042,1538352935000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",281,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,265.50812,Under The Bridge (Album Version),Red Hot Chili Peppers,2018-10-01 00:15:35,2018-08-08 13:22:21
2366,200,M,Shlok,paid,Johnson,1749042,1538353200000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",282,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,471.69261,Circlesong 6,Bobby McFerrin,2018-10-01 00:20:00,2018-08-08 13:22:21


In [121]:
df. shape

(17499636, 19)

In [ ]:
df_test=df.copy()
df_test["time_converted"]=pd.to_datetime(df["ts"], unit="ms")
df_test.head()
# On voit bien ici que ts et time, représentent la même information dans deux formats différents. 
#ts représente le Unix timestamp, alors que time est déjà dans le format de date que nous connaissons



MemoryError: Unable to allocate 1.56 GiB for an array with shape (12, 17499636) and data type object

In [124]:
df_test["diff"]=df_test["registration"]-df["time"]
df["diff"].describe()

KeyError: 'diff'

In [116]:
df["auth"].value_counts()

auth
Logged In    17495365
Cancelled        4271
Name: count, dtype: int64

In [117]:
df["status"].value_counts()

status
200    16020693
307     1461649
404       17294
Name: count, dtype: int64

In [118]:
df.dtypes

status                    int64
gender                   object
firstName                object
level                    object
lastName                 object
userId                   object
ts                        int64
auth                     object
page                     object
sessionId                 int64
location                 object
itemInSession             int64
userAgent                object
method                   object
length                  float64
song                     object
artist                   object
time             datetime64[us]
registration     datetime64[us]
dtype: object

### A better understanding of the different variables, let's summarize it here: 
- status: it represents the technical result of a request (200= good, 307=warning, 400=bad)

- ts: it represents the timestamp (the exact moment when the event was registered, need to be transformed with "to_date_time")

- auth: it is the status of authentification (logg In or cancelled)

- sessionId: unique key of a user session

- itemInSession: the sequential number of the event within the session

- page: represent the page visited by the user



### Importation des fonctions utiles pour le préprocessing

In [54]:
from page_features import create_page_features
from method_features import get_method_features
from level_features import get_level_features
from song_artist_itemInSession_features import create_df_autres
from gender_feature import gender_feature_ont_hot
from status import num_status
from time_features import total_number_of_sessions, usage_metrices
from location_features import gap_encoder_location, location_state_one_hot_encoded
from length_features import get_length_features


### Preprocessing à proprement parler

In [33]:
df_level=get_level_features(df)
df_level.shape
df_level.head()


,level
userId,
1000025,1
1000035,1
1000083,1
1000103,1
1000164,1


In [16]:
df_page=create_page_features(df)
df_page.shape

(19140, 19)

In [34]:
df_page.head()

page,About,Add Friend,Add to Playlist,Cancel,Cancellation Confirmation,Downgrade,Error,Help,Home,Logout,NextSong,Roll Advert,Save Settings,Settings,Submit Downgrade,Submit Upgrade,Thumbs Down,Thumbs Up,Upgrade
userId,,,,,,,,,,,,,,,,,,,
1000025,1,30,53,1,1,18,1,8,77,26,1662,7,3,8,0,1,13,94,1
1000035,2,23,27,0,0,7,1,5,54,19,1266,6,1,7,0,1,15,117,5
1000083,0,6,8,1,1,2,0,2,21,14,501,8,0,5,0,1,2,21,3
1000103,0,0,1,0,0,1,0,0,6,2,57,3,0,0,0,1,1,2,1
1000164,2,17,24,0,0,10,1,4,40,11,847,20,1,5,0,1,6,38,1


In [ ]:
df_method=get_method_features(df)
df_method.shape


(19140, 1)

In [35]:
df_method.head()

,method
userId,
1000025,1
1000035,1
1000083,1
1000103,1
1000164,1


In [19]:
df_autres=create_df_autres(df)
df_autres.shape

(19140, 3)

In [36]:
df_autres.head()

,song,artist,itemInSession
userId,,,
1000025,1468,1162,486
1000035,1154,916,228
1000083,478,427,171
1000103,57,56,53
1000164,778,660,215


In [20]:
df_gender=gender_feature_ont_hot(df)
df_gender.shape

(19140, 2)

In [38]:
df_gender.set_index('userId', inplace=True)
df_gender.head()

,gender
userId,
1749042,True
1563081,False
1697168,False
1222580,True
1714398,False


In [22]:
df_status=num_status(df)
df_status.shape

(19140, 3)

In [41]:
 
df_status.head()


status,200,307,404
userId,,,
1000025,1836.0,168.0,1.0
1000035,1379.0,176.0,1.0
1000083,551.0,45.0,0.0
1000103,69.0,6.0,0.0
1000164,953.0,74.0,1.0


In [23]:
df_time=total_number_of_sessions(df)
df_time.shape

(19140, 2)

In [43]:
df_time.set_index('userId', inplace=True)   
df_time.head()

,total_number_of_sessions
userId,
1000025,17
1000035,21
1000083,11
1000103,3
1000164,15


In [ ]:
df_usage=usage_metrices(df)
df_usage.shape

(19140, 28)

In [29]:
df_usage.head()

week_number,40,41,42,43,44,46,47,delta_week_47_40,delta_week_47_41,delta_week_47_42,...,%_delta_week_47_46,mean_weekly_usage,median_weekly_usage,var_weekly_usage,mean_weekly_delta,median_weekly_delta,var_weekly_delta,mean_weekly_delta_%,median_weekly_delta_%,var_weekly_delta_%
userId,,,,,,,,,,,,,,,,,,,,,
1000025,34944.533034,13464.054230,22910.624852,0.000000,0.000000,0.000000,0.000000,-34944.533034,-13464.054230,-22910.624852,...,0.000000,10188.458874,0.000000,2.001108e+08,-11886.535353,-6732.027115,2.159119e+08,-50.000000,-50.00000,3.000000e+03
1000035,67.031924,2324.556103,9592.823503,7576.029159,6107.531961,10464.205455,0.000000,-67.031924,-2324.556103,-9592.823503,...,-100.000000,5161.739729,6107.531961,1.918720e+07,-6022.029684,-6841.780560,1.680781e+07,-100.000000,-100.00000,0.000000e+00
1000083,4643.794678,13181.801179,1151.246826,0.000000,0.000000,0.000000,0.000000,-4643.794678,-13181.801179,-1151.246826,...,0.000000,2710.977526,0.000000,2.420071e+07,-3162.807114,-575.623413,2.732600e+07,-50.000000,-50.00000,3.000000e+03
1000103,2442.261705,0.000000,0.000000,300.171497,100.057166,0.000000,0.000000,-2442.261705,0.000000,0.000000,...,0.000000,406.070053,0.000000,8.184177e+05,-473.748395,-50.028583,9.436263e+05,-50.000000,-50.00000,3.000000e+03
1000164,51.911756,4129.249891,7189.057606,1122.994641,628.672950,5149.720106,14435.430277,14383.518521,10306.180386,7246.372671,...,180.314852,4672.433890,4129.249891,2.541202e+07,11390.162453,11809.308011,8.253995e+06,5269.991615,717.51518,1.214819e+08


In [ ]:
df_location_state=location_state_one_hot_encoded(df)
df_location_state.shape


(19140, 52)

In [46]:

df_location_state.shape

(19140, 51)

In [56]:
df_length=df.groupby('userId').agg({"length": "sum"}).reset_index().set_index('userId')

In [58]:
df_length.head()

,length
userId,
1000025,417296.59169
1000035,310364.86590
1000083,122606.27093
1000103,13554.73009
1000164,209060.65753


In [60]:
df_final=pd.concat([df_level, df_page, df_method, df_autres, df_gender, df_status, df_time, df_usage, df_location_state, df_length], axis=1)  

In [ ]:
df_final.isna().any(axis=1).sum()

3377

### Première selection de features

Ici, je fais une première selection de variables sur la base de la pertinence des variables. Les variables retenues sont:
- page
- level
- song
- artist
- location
- length
- auth
- gender

In [63]:
df_final.columns

Index(['level', 'About', 'Add Friend', 'Add to Playlist', 'Cancel',
       'Cancellation Confirmation', 'Downgrade', 'Error', 'Help', 'Home',
       ...
       'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY', 'length'],
      dtype='object', length=109)

In [ ]:
df_test=df_final.dropna(axis=0)

In [67]:
df_test["Cancellation Confirmation"].value_counts()/df_test.shape[0]

Cancellation Confirmation
0    0.73533
1    0.26467
Name: count, dtype: float64

In [68]:
df_final["Cancellation Confirmation"].value_counts()/df_final.shape[0]  

Cancellation Confirmation
0    0.776855
1    0.223145
Name: count, dtype: float64

In [69]:
# suppression des NaN dans le dataset final
df_final=df_test

In [76]:
df_final.drop("Cancel", axis=1, inplace=True)

In [77]:
df_final.rename(columns={"Cancellation Confirmation": "churned"}, inplace=True)

In [78]:
df_final.columns

Index(['level', 'About', 'Add Friend', 'Add to Playlist', 'churned',
       'Downgrade', 'Error', 'Help', 'Home', 'Logout',
       ...
       'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY', 'length'],
      dtype='object', length=108)

In [79]:
df_final.head()

,level,About,Add Friend,Add to Playlist,churned,Downgrade,Error,Help,Home,Logout,...,TN,TX,UT,VA,VT,WA,WI,WV,WY,length
userId,,,,,,,,,,,,,,,,,,,,,
1000025,1,1,30,53,1,18,1,8,77,26,...,0,0,0,0,0,0,0,0,0,417296.59169
1000035,1,2,23,27,0,7,1,5,54,19,...,0,0,0,0,0,0,0,0,0,310364.86590
1000083,1,0,6,8,1,2,0,2,21,14,...,0,0,0,0,0,0,0,0,0,122606.27093
1000103,1,0,0,1,0,1,0,0,6,2,...,0,0,0,0,0,0,0,0,0,13554.73009
1000164,1,2,17,24,0,10,1,4,40,11,...,0,0,0,0,0,0,0,0,0,209060.65753


In [80]:
df_final.dtypes

level                int32
About                int64
Add Friend           int64
Add to Playlist      int64
churned              int64
                    ...   
WA                   int64
WI                   int64
WV                   int64
WY                   int64
length             float64
Length: 108, dtype: object

In [90]:
model=RandomForestClassifier(n_estimators=200, random_state=42 )

In [ ]:
unique_users=df_final.index  
users_label=df_final['churned']

train_ids, test_ids=train_test_split(unique_users, test_size=0.2, random_state=42, stratify=users_label)
X_train=df_final.loc[train_ids].drop('churned', axis=1)
X_test=df_final.loc[test_ids].drop("churned", axis=1)
y_train=df_final.loc[train_ids, "churned"]
y_test=df_final.loc[test_ids, "churned"]

In [92]:
model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [93]:
y_pred=model.predict(X_test)

In [94]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.95      0.92      2318
           1       0.82      0.69      0.75       835

    accuracy                           0.88      3153
   macro avg       0.86      0.82      0.84      3153
weighted avg       0.88      0.88      0.88      3153



In [95]:
print(f"AUC-ROC-Score: {roc_auc_score(y_test, y_pred)}")

AUC-ROC-Score: 0.8199110837858364


In [102]:
confusion_matrix(y_test, y_pred)

array([[2191,  127],
       [ 255,  580]], dtype=int64)